# MedScope AI — Full System Inference on Colab

This notebook pulls code from GitHub, loads your trained weights from Google Drive, and starts both the Backend API and the Next.js Frontend using localtunnel.

In [ ]:
#@title 1. Runtime and configuration
import os, sys
REPO_URL = "https://github.com/prey801/finalyearproject.git"
BRANCH = "main"
PROJECT_DIR = "/content/finalyearproject"
DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/medscope-ai/weights"

SUPABASE_URL = "" #@param {type:"string"}
SUPABASE_ANON_KEY = "" #@param {type:"string"}

GITHUB_TOKEN = "" #@param {type:"string"}


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Clone Repository
!git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd $PROJECT_DIR

In [ ]:
#@title 4. Install localtunnel & SAM2
!npm install -g localtunnel
%cd $PROJECT_DIR
!wget -q -O sam2_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
print("SAM2 weights downloaded!")

In [ ]:
#@title 5. Install Backend & Frontend Dependencies
!apt-get update -qq
!apt-get install -y -qq redis-server postgresql libgl1
!python -m pip install -q -U pip setuptools wheel
!python -m pip install -q -r backend/requirements.txt
!python -m pip install -q -r models/requirements.txt
!python -m pip install -q pytest qdrant-client nest_asyncio pyngrok
%cd $PROJECT_DIR/frontend
!npm install
%cd $PROJECT_DIR

In [ ]:
#@title 6. Load Trained Weights from Drive
import shutil
from pathlib import Path
src_dir = Path(DRIVE_WEIGHTS_DIR)
dst_dir = Path(PROJECT_DIR) / "models/weights"
dst_dir.mkdir(parents=True, exist_ok=True)
for p in src_dir.glob("*.pt"):
    print(f"Copying {p.name}...")
    shutil.copy2(p, dst_dir / p.name)
for p in src_dir.glob("*.pth"):
    print(f"Copying {p.name}...")
    shutil.copy2(p, dst_dir / p.name)
print("Weights loaded!")

In [ ]:
#@title 7. Start Infrastructure (Redis, Postgres, Qdrant)
!service redis-server start
!service postgresql start

# Set up PostgreSQL
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';" || true
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;" || true

# Set up Qdrant
import subprocess, time
from pathlib import Path
%cd $PROJECT_DIR
if not Path("qdrant").exists():
    !wget -q https://github.com/qdrant/qdrant/releases/download/v1.7.0/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz

subprocess.Popen(["./qdrant"])
time.sleep(3)
print("Infrastructure services started.")

In [ ]:
#@title 8. Run the Entire System
%cd $PROJECT_DIR
import os, subprocess, time
from pathlib import Path

# Set environment variables for backend
if "GITHUB_TOKEN" in globals() and GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

os.environ['YOLO_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/yolov11_malaria_best.pt")
os.environ['SWIN_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/swin_malaria_best.pth")
os.environ['SAM2_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "sam2_hiera_small.pt")
os.environ['ALLOWED_ORIGINS'] = "*"

# 1. Start Backend
print("Starting backend...")
backend_process = subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])
frontend_process.wait()
print("Starting frontend server...")
frontend_process = subprocess.Popen(["npm", "start"], cwd=f"{PROJECT_DIR}/frontend")
time.sleep(5)

backend_lt = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, text=True)
backend_url_line = backend_lt.stdout.readline().strip()
backend_url = backend_url_line.split(" ")[-1] if backend_url_line else "http://localhost:8000"
print(f"Backend exposed at: {backend_url}")

# 2. Configure Frontend with Backend URL
with open(f"{PROJECT_DIR}/frontend/.env.local", "w") as f:
    f.write(f"NEXT_PUBLIC_API_URL={backend_url}\n")
    if "SUPABASE_URL" in globals() and SUPABASE_URL:
        f.write(f"NEXT_PUBLIC_SUPABASE_URL={SUPABASE_URL}\n")
        f.write(f"NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY={SUPABASE_ANON_KEY}\n")
        f.write(f"NEXT_PUBLIC_SUPABASE_ANON_KEY={SUPABASE_ANON_KEY}\n")

# 3. Start Frontend
print("Starting frontend...")
frontend_process = subprocess.Popen(["npm", "run", "build"], cwd=f"{PROJECT_DIR}/frontend")
frontend_process.wait()
print("Starting frontend server...")
frontend_process = subprocess.Popen(["npm", "start"], cwd=f"{PROJECT_DIR}/frontend")
time.sleep(5)

frontend_lt = subprocess.Popen(["lt", "--port", "3000"], stdout=subprocess.PIPE, text=True)
frontend_url_line = frontend_lt.stdout.readline().strip()
frontend_url = frontend_url_line.split(" ")[-1] if frontend_url_line else "http://localhost:3000"

print("\n" + "="*60)
print(f"🚀 SYSTEM IS RUNNING!")
print(f"🌍 Access the Frontend Web App here: {frontend_url}")
print(f"⚙️ Backend API is at: {backend_url}")
print("="*60 + "\n")

try:
    frontend_process.wait()
except KeyboardInterrupt:
    print("\nShutting down...")
    backend_process.terminate()
    frontend_process.terminate()
    backend_lt.terminate()
    frontend_lt.terminate()